# Infra-FM: Asset Classification

Runs two classification experiments back to back and produces a comparison table:
1. **Baseline** — random init encoder, fine-tuned
2. **Pretrained** — SimCLR encoder from central america pretraining, linear probe
3. **Pretrained fine-tuned** — same checkpoint, encoder unfrozen

**Before running:**
- `My Drive/infra_fm/datasets/dataset_central-america_stac_v1/` — completed dataset
- `My Drive/infra_fm/results/pretrain_central_america/best.pt` — pretrained checkpoint
- `My Drive/infra_fm/code/infra_fm_curation.zip` — curation code (for any imports)

**Runtime:** Set to GPU (T4) via Runtime → Change runtime type

## 1. Mount Drive + check GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mounted at /content/drive
GPU available: True
GPU: Tesla T4
Memory: 15.6 GB


## 2. Install dependencies

In [2]:
%%capture
!pip install scipy opencv-python-headless

## 3. Set up code

In [3]:
import os, sys, shutil, zipfile
from pathlib import Path

DRIVE_ROOT  = '/content/drive/MyDrive/infra_fm'
CODE_ZIP    = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO  = '/content/infra_fm_clean'

# Extract with Windows backslash fix
os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP, 'r') as z:
    for member in z.namelist():
        clean_path = member.replace('\\', '/')
        target = os.path.join(EXTRACT_TO, clean_path)
        if clean_path.endswith('/'):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src, open(target, 'wb') as dst:
                dst.write(src.read())

# Find the root that contains downstream/
code_root = None
for root, dirs, files in os.walk(EXTRACT_TO):
    if 'downstream' in dirs:
        code_root = root
        break

if code_root is None:
    raise RuntimeError('Could not find downstream/ folder in extracted zip')

sys.path.insert(0, code_root)
os.chdir(code_root)
print(f'Code root: {code_root}')
print(f'Contents: {sorted(os.listdir(code_root))}')

# Verify key imports
from downstream.common.models import EncoderBackbone, SimCLRModel, LinearClassifier
from downstream.common.transforms import build_eval_transform
from downstream.common.utils import choose_device, ensure_dir, save_checkpoint, save_json, set_seed
from downstream.asset_classification.datasets import AssetClassificationDataset
print('All imports OK')

Code root: /content/infra_fm_clean/infra_fm_code_only
Contents: ['.gitignore', 'README.md', 'additional_info', 'caadd_asset_table_code_test.py', 'curation', 'downstream', 'hypergraphs', 'inspect_dataset.py', 'outputs', 'pretraining', 'testing']
All imports OK


In [17]:
import os
for root, dirs, files in os.walk('/content'):
    if 'downstream' in dirs:
        print('Code root:', root)
        break

Code root: /content/infra_fm_clean/infra_fm_code_only


## 4. Configuration

In [4]:
# --- Paths ---
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/dataset_central-america_stac_v1'
CHECKPOINT   = f'{DRIVE_ROOT}/results/pretrain_central_america/best.pt'
RESULTS_DIR  = f'{DRIVE_ROOT}/results/classification'

os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Training settings ---
BAND_INDICES    = '0,1,2,3,4,5,6,7,8,9'  # 10 bands: sentinel2_ms + sentinel1 + landsat_thermal
BACKBONE        = 'resnet18'
EPOCHS          = 30                       # more epochs on GPU since it's fast
BATCH_SIZE      = 32                       # larger batch on GPU
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4
TRAIN_FRACTION  = 0.8
SEED            = 42
TAIL_EPOCHS     = 5
IMAGE_SIZE      = 224

# Verify dataset and checkpoint exist
print(f'Dataset exists:    {Path(DATASET_ROOT).exists()}')
print(f'Checkpoint exists: {Path(CHECKPOINT).exists()}')
print(f'Results dir:       {RESULTS_DIR}')

# Check band count from manifest
import json
manifest = json.load(open(f'{DATASET_ROOT}/manifest.json'))
sample_shape = manifest['records'][0].get('image_shape', [])
print(f'\nDataset: {manifest["n_tiles"]} tiles')
print(f'Modalities: {manifest.get("modalities")}')
print(f'Sample shape (C,H,W): {sample_shape}')
n_bands = sample_shape[0] if sample_shape else '?'
print(f'Bands in tiles: {n_bands}')
print(f'Band indices selected: {BAND_INDICES}')

Dataset exists:    True
Checkpoint exists: True
Results dir:       /content/drive/MyDrive/infra_fm/results/classification

Dataset: 1782 tiles
Modalities: ['sentinel1', 'sentinel2_ms', 'landsat_thermal']
Sample shape (C,H,W): [10, 61, 61]
Bands in tiles: 10
Band indices selected: 0,1,2,3,4,5,6,7,8,9


## 5. Shared training utilities

In [5]:
import csv
import random
import warnings
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset


def split_indices(n, train_fraction, seed):
    idxs = list(range(n))
    random.Random(seed).shuffle(idxs)
    cut = max(1, int(n * train_fraction))
    return idxs[:cut], idxs[cut:]


def compute_class_weights(dataset, train_indices, device):
    counts = [0] * len(dataset.label_space.classes)
    for idx in train_indices:
        counts[dataset[idx]['label'].item()] += 1
    n_classes = len(counts)
    total = sum(counts)
    weights = [total / (n_classes * c) if c > 0 else 0.0 for c in counts]
    print('Class weights (inverse frequency):')
    for cls, w, c in zip(dataset.label_space.classes, weights, counts):
        print(f'  {cls.split(".")[-1]:35s} count={c:>5}  weight={w:.3f}')
    return torch.tensor(weights, dtype=torch.float32, device=device)


def evaluate(encoder, head, loader, device, classes=None):
    if not loader.dataset:
        return 0.0, None
    encoder.eval(); head.eval()
    correct, total = 0, 0
    class_correct = [0] * len(classes) if classes else None
    class_total   = [0] * len(classes) if classes else None
    with torch.no_grad():
        for batch in loader:
            x = batch['image'].to(device)
            y = batch['label'].to(device)
            pred = head(encoder(x)).argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.numel()
            if classes:
                for i in range(len(classes)):
                    mask = (y == i)
                    class_correct[i] += (pred[mask] == y[mask]).sum().item()
                    class_total[i]   += mask.sum().item()
    overall = correct / max(total, 1)
    per_class = (
        {cls: round(class_correct[i] / max(class_total[i], 1), 4)
         for i, cls in enumerate(classes)}
        if classes else None
    )
    return overall, per_class


def run_experiment(name, dataset_root, band_indices, backbone, epochs,
                   batch_size, lr, wd, train_fraction, seed, tail_epochs,
                   image_size, device, output_dir,
                   checkpoint=None, freeze_encoder=False):
    print(f'\n{"=" * 60}')
    print(f'EXPERIMENT: {name}')
    print(f'  checkpoint={checkpoint}')
    print(f'  freeze_encoder={freeze_encoder}')
    print('=' * 60)

    set_seed(seed)
    out = ensure_dir(output_dir)

    dataset = AssetClassificationDataset(
        dataset_root=dataset_root,
        transform=build_eval_transform(image_size),
        band_indices=band_indices,
    )
    classes = dataset.label_space.classes
    print(f'Samples: {len(dataset)} | Classes ({len(classes)}): {classes}')

    train_idxs, val_idxs = split_indices(len(dataset), train_fraction, seed)
    train_loader = DataLoader(Subset(dataset, train_idxs), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(Subset(dataset, val_idxs),   batch_size=batch_size, shuffle=False)
    print(f'Train: {len(train_idxs)} | Val: {len(val_idxs)}')

    # Build model
    in_channels = len([int(x) for x in band_indices.split(',')])
    if checkpoint and Path(checkpoint).exists():
        ckpt = torch.load(checkpoint, map_location='cpu')
        config = ckpt.get('config', {}) if isinstance(ckpt, dict) else {}
        model = SimCLRModel(
            backbone_name=config.get('backbone_name', backbone),
            pretrained_backbone=False,
            projection_dim=config.get('projection_dim', 128),
            in_channels=in_channels,
        )
        state_dict = (
            ckpt.get('model_state') or ckpt.get('model_state_dict')
            or ckpt.get('state_dict') or ckpt
        )
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        encoder = model.backbone
        encoder.feature_dim = model.feature_dim
        print(f'Loaded checkpoint: {len(state_dict) - len(missing)}/{len(state_dict)} keys matched')
    else:
        encoder = EncoderBackbone(backbone, pretrained=False, in_channels=in_channels)
        print('Random init encoder')

    head = LinearClassifier(encoder.feature_dim, len(classes))
    encoder.to(device); head.to(device)

    if freeze_encoder:
        for p in encoder.parameters():
            p.requires_grad = False
        print('Encoder frozen (linear probe)')

    params = list(head.parameters()) + [p for p in encoder.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=lr, weight_decay=wd)
    class_weights = compute_class_weights(dataset, train_idxs, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_acc = -1.0
    val_history = []

    history_path = out / 'metrics.csv'
    with history_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_acc'])
        writer.writeheader()
        for epoch in range(1, epochs + 1):
            encoder.train(); head.train()
            losses = []
            for batch in train_loader:
                x = batch['image'].to(device)
                y = batch['label'].to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(head(encoder(x)), y)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            val_acc, _ = evaluate(encoder, head, val_loader, device)
            val_history.append(val_acc)
            train_loss = sum(losses) / max(len(losses), 1)
            writer.writerow({'epoch': epoch,
                             'train_loss': round(train_loss, 6),
                             'val_acc': round(val_acc, 6)})
            f.flush()
            print(f'Epoch {epoch:03d}/{epochs:03d} | loss={train_loss:.4f} | val_acc={val_acc:.4f}')
            if val_acc > best_acc:
                best_acc = val_acc
                save_checkpoint(out / 'checkpoint_best.pt', {
                    'encoder_state': encoder.state_dict(),
                    'head_state':    head.state_dict(),
                    'classes':       classes,
                })

    tail_n    = min(tail_epochs, len(val_history))
    tail_vals = val_history[-tail_n:]
    tail_mean = sum(tail_vals) / len(tail_vals)
    tail_std  = (sum((v - tail_mean) ** 2 for v in tail_vals) / len(tail_vals)) ** 0.5

    _, per_class = evaluate(encoder, head, val_loader, device, classes=classes)

    summary = {
        'experiment':    name,
        'best_val_acc':  round(best_acc, 4),
        'tail_mean_acc': round(tail_mean, 4),
        'tail_std_acc':  round(tail_std, 4),
        'tail_epochs':   tail_n,
        'per_class_acc': per_class,
        'classes':       classes,
    }
    save_json(out / 'results_summary.json', summary)

    print(f'\nBest val_acc:  {best_acc:.4f}')
    print(f'Tail mean acc: {tail_mean:.4f} ± {tail_std:.4f} (last {tail_n} epochs)')
    print('Per-class accuracy:')
    if per_class:
        for cls, acc in per_class.items():
            print(f'  {cls.split(".")[-1]:35s} {acc:.4f}')

    return summary


print('Utilities loaded.')

Utilities loaded.


## 6. Experiment 1 — Random init baseline (fine-tuned)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

result_baseline = run_experiment(
    name           = 'random_init_finetuned',
    dataset_root   = DATASET_ROOT,
    band_indices   = BAND_INDICES,
    backbone       = BACKBONE,
    epochs         = EPOCHS,
    batch_size     = BATCH_SIZE,
    lr             = LEARNING_RATE,
    wd             = WEIGHT_DECAY,
    train_fraction = TRAIN_FRACTION,
    seed           = SEED,
    tail_epochs    = TAIL_EPOCHS,
    image_size     = IMAGE_SIZE,
    device         = device,
    output_dir     = f'{RESULTS_DIR}/baseline',
    checkpoint     = None,
    freeze_encoder = False,
)

Using device: cuda

EXPERIMENT: random_init_finetuned
  checkpoint=None
  freeze_encoder=False
Samples: 1782 | Classes (3): ['energy.distribution.substation', 'energy.distribution.substation_untyped', 'energy.transmission.substation']
Train: 1425 | Val: 357
Random init encoder
Class weights (inverse frequency):
  substation                          count=  158  weight=3.006
  substation_untyped                  count=  921  weight=0.516
  substation                          count=  346  weight=1.373
Epoch 001/030 | loss=1.2606 | val_acc=0.3697
Epoch 002/030 | loss=1.0891 | val_acc=0.4426
Epoch 003/030 | loss=1.0696 | val_acc=0.4454
Epoch 004/030 | loss=1.0668 | val_acc=0.3081
Epoch 005/030 | loss=1.0791 | val_acc=0.4846
Epoch 006/030 | loss=1.0728 | val_acc=0.3473
Epoch 007/030 | loss=1.0486 | val_acc=0.3725
Epoch 008/030 | loss=1.0355 | val_acc=0.3950
Epoch 009/030 | loss=1.0337 | val_acc=0.2241
Epoch 010/030 | loss=1.0416 | val_acc=0.4846
Epoch 011/030 | loss=1.0626 | val_acc=0.5154


## 7. Experiment 2 — Pretrained encoder, linear probe (frozen)

In [7]:
result_linear_probe = run_experiment(
    name           = 'simclr_pretrained_linear_probe',
    dataset_root   = DATASET_ROOT,
    band_indices   = BAND_INDICES,
    backbone       = BACKBONE,
    epochs         = EPOCHS,
    batch_size     = BATCH_SIZE,
    lr             = LEARNING_RATE,
    wd             = WEIGHT_DECAY,
    train_fraction = TRAIN_FRACTION,
    seed           = SEED,
    tail_epochs    = TAIL_EPOCHS,
    image_size     = IMAGE_SIZE,
    device         = device,
    output_dir     = f'{RESULTS_DIR}/pretrained_linear_probe',
    checkpoint     = CHECKPOINT,
    freeze_encoder = True,
)


EXPERIMENT: simclr_pretrained_linear_probe
  checkpoint=/content/drive/MyDrive/infra_fm/results/pretrain_central_america/best.pt
  freeze_encoder=True
Samples: 1782 | Classes (3): ['energy.distribution.substation', 'energy.distribution.substation_untyped', 'energy.transmission.substation']
Train: 1425 | Val: 357
Loaded checkpoint: 124/124 keys matched
Encoder frozen (linear probe)
Class weights (inverse frequency):
  substation                          count=  158  weight=3.006
  substation_untyped                  count=  921  weight=0.516
  substation                          count=  346  weight=1.373
Epoch 001/030 | loss=1.1735 | val_acc=0.3782
Epoch 002/030 | loss=1.0739 | val_acc=0.4678
Epoch 003/030 | loss=1.0817 | val_acc=0.2577
Epoch 004/030 | loss=1.1032 | val_acc=0.3725
Epoch 005/030 | loss=1.0607 | val_acc=0.5182
Epoch 006/030 | loss=1.0848 | val_acc=0.3109
Epoch 007/030 | loss=1.0608 | val_acc=0.3109
Epoch 008/030 | loss=1.1105 | val_acc=0.2157
Epoch 009/030 | loss=1.0753 

## 8. Experiment 3 — Pretrained encoder, fine-tuned (unfrozen)

In [8]:
result_finetuned = run_experiment(
    name           = 'simclr_pretrained_finetuned',
    dataset_root   = DATASET_ROOT,
    band_indices   = BAND_INDICES,
    backbone       = BACKBONE,
    epochs         = EPOCHS,
    batch_size     = BATCH_SIZE,
    lr             = LEARNING_RATE,
    wd             = WEIGHT_DECAY,
    train_fraction = TRAIN_FRACTION,
    seed           = SEED,
    tail_epochs    = TAIL_EPOCHS,
    image_size     = IMAGE_SIZE,
    device         = device,
    output_dir     = f'{RESULTS_DIR}/pretrained_finetuned',
    checkpoint     = CHECKPOINT,
    freeze_encoder = False,
)


EXPERIMENT: simclr_pretrained_finetuned
  checkpoint=/content/drive/MyDrive/infra_fm/results/pretrain_central_america/best.pt
  freeze_encoder=False
Samples: 1782 | Classes (3): ['energy.distribution.substation', 'energy.distribution.substation_untyped', 'energy.transmission.substation']
Train: 1425 | Val: 357
Loaded checkpoint: 124/124 keys matched
Class weights (inverse frequency):
  substation                          count=  158  weight=3.006
  substation_untyped                  count=  921  weight=0.516
  substation                          count=  346  weight=1.373
Epoch 001/030 | loss=1.3127 | val_acc=0.2605
Epoch 002/030 | loss=1.1781 | val_acc=0.4930
Epoch 003/030 | loss=1.1290 | val_acc=0.3417
Epoch 004/030 | loss=1.1101 | val_acc=0.3081
Epoch 005/030 | loss=1.1198 | val_acc=0.3165
Epoch 006/030 | loss=1.0941 | val_acc=0.4342
Epoch 007/030 | loss=1.1035 | val_acc=0.4230
Epoch 008/030 | loss=1.0926 | val_acc=0.2521
Epoch 009/030 | loss=1.0886 | val_acc=0.4482
Epoch 010/030 |

## 9. Comparison table

In [16]:
import json

results = [result_baseline, result_linear_probe, result_finetuned]

print('\n' + '=' * 70)
print('COMPARISON TABLE')
print('=' * 70)
print(f'{"Experiment":40s} {"Best":>6} {"Tail mean":>10} {"Tail std":>10}')
print('-' * 70)
for r in results:
    name = r['experiment'].replace('_', ' ')
    print(f'{name:40s} {r["best_val_acc"]:>6.4f} {r["tail_mean_acc"]:>10.4f} {r["tail_std_acc"]:>10.4f}')

print('\nPer-class accuracy (final epoch):')
classes = results[0]['classes']
short_classes = [
    c.replace("energy.", "").replace("distribution.", "dx_").replace("transmission.", "tx_")
    for c in classes
]
header = f'{"":40s}' + ''.join(f'{c:>20s}' for c in short_classes)
print(header)
print('-' * (30 + 30 * len(classes)))
for r in results:
    name = r['experiment'].replace('_', ' ')
    row = f'{name:40s}'
    if r['per_class_acc']:
        for cls in classes:
            row += f'{r["per_class_acc"].get(cls, 0.0):>20.4f}'
    print(row)

# Save comparison to Drive
comparison = {
    'runs': results,
    'dataset': 'central-america_stac_v1',
    'epochs': EPOCHS,
    'band_indices': BAND_INDICES,
}
out_path = f'{RESULTS_DIR}/comparison_table.json'
with open(out_path, 'w') as f:
    json.dump(comparison, f, indent=2)
print(f'\nSaved comparison to {out_path}')


COMPARISON TABLE
Experiment                                 Best  Tail mean   Tail std
----------------------------------------------------------------------
random init finetuned                    0.5406     0.3832     0.0890
simclr pretrained linear probe           0.5182     0.4269     0.0212
simclr pretrained finetuned              0.5266     0.3832     0.0588

Per-class accuracy (final epoch):
                                               dx_substationdx_substation_untyped       tx_substation
------------------------------------------------------------------------------------------------------------------------
random init finetuned                                 0.4884              0.1593              0.6932
simclr pretrained linear probe                        0.3256              0.3407              0.6023
simclr pretrained finetuned                           0.2326              0.1372              0.7614

Saved comparison to /content/drive/MyDrive/infra_fm/results/classific